# ProBat Insight - Run Backend from Google Colab

This notebook:
1. Clones the repo and installs all backend + ML dependencies
2. Lets you set your MongoDB URI and secret key
3. Starts the FastAPI server
4. Creates a public **ngrok** tunnel URL
5. You use that URL in your React frontend instead of `localhost:8000`

> **Requirement:** Free [ngrok account](https://ngrok.com) - grab your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken

## 1  Clone Repository & Install Dependencies

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/Anjalee-jay/Probat-Insight.git'
REPO_DIR = '/content/Probat-Insight'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(os.path.join(REPO_DIR, 'backend'))

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi==0.115.12',
    'uvicorn[standard]==0.34.0',
    'pymongo==4.12.0',
    'python-dotenv==1.1.0',
    'python-jose[cryptography]==3.3.0',
    'passlib==1.7.4',
    'email-validator==2.2.0',
    'mediapipe==0.10.18',
    'opencv-python-headless==4.10.0.84',
    'numpy==1.26.4',
    'scikit-learn==1.5.2',
    'pandas==2.2.3',
    'joblib==1.4.2',
    'Pillow==10.4.0',
    'python-multipart==0.0.9',
    'pyngrok',
], check=True)

print('All dependencies installed.')

## 2  Configure Environment Variables

In [ ]:
# -----------------------------------------------------------------
#  Fill in your values below before running this cell
# -----------------------------------------------------------------

MONGODB_URL = 'mongodb+srv://<user>:<password>@<cluster>.mongodb.net/<dbname>?retryWrites=true&w=majority'
SECRET_KEY  = 'change-me-to-a-long-random-string'
NGROK_TOKEN = 'paste-your-ngrok-authtoken-here'

env_content = (
    f'MONGODB_URL={MONGODB_URL}\n'
    f'SECRET_KEY={SECRET_KEY}\n'
    'ALGORITHM=HS256\n'
    'ACCESS_TOKEN_EXPIRE_MINUTES=60\n'
    'CORS_ORIGINS=*\n'
)

with open('/content/Probat-Insight/backend/.env', 'w') as f:
    f.write(env_content)

print('.env written - do NOT share this notebook with credentials inside.')

## 3  Start FastAPI Server + ngrok Tunnel

In [ ]:
import subprocess
import sys
import threading
import time

from pyngrok import conf, ngrok  # type: ignore[import]

# Authenticate ngrok
conf.get_default().auth_token = NGROK_TOKEN

# Start uvicorn in a background thread
def run_server() -> None:
    subprocess.run(
        [sys.executable, '-m', 'uvicorn', 'app.main:app',
         '--host', '0.0.0.0', '--port', '8000'],
        cwd='/content/Probat-Insight/backend'
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(4)  # wait for server to be ready

# Open ngrok tunnel
tunnel = ngrok.connect(8000, bind_tls=True)
public_url: str = tunnel.public_url

print('=' * 60)
print(f'  Backend URL : {public_url}')
print(f'  API Docs    : {public_url}/docs')
print(f'  Health      : {public_url}/api/health')
print('=' * 60)
print()
print('Set this in your React frontend .env:')
print(f'  REACT_APP_API_URL={public_url}')

## 4  Verify the Server is Working

Run this to confirm the API is reachable.

In [ ]:
import json

import requests

r = requests.get(f'{public_url}/api/health')
print(json.dumps(r.json(), indent=2))

## 5  Point Your React Frontend to the Tunnel

In your local project, add a `.env` file at the React root:

```
# .env
REACT_APP_API_URL=https://xxxx-xx-xx-xxx-xx.ngrok-free.app
```

Then restart your React dev server: `npm start`

> **Note:** The ngrok URL changes every time you restart this notebook (free tier). Update `.env` each session.

## 6  Stop the Server (when done)

In [ ]:
from pyngrok import ngrok  # type: ignore[import]

ngrok.kill()
print('ngrok tunnel closed.')